# ETF Explorer (v1)

Discovery view over Stashaway's full ETF Explorer offering (~98 ETFs).
Computes multi-window metrics (1Y/3Y/5Y) + correlation with your combined book.
Renders to `reports/etf_explorer.html`.

In [1]:
from datetime import date
from pathlib import Path

from hailmary.allocation.book_config import MGMT_FEES_ANNUAL, ROLES
from hailmary.allocation.etf_explorer import build_etf_explorer, render_etf_explorer_report
from hailmary.allocation.portfolios import from_parsed
from hailmary.allocation.statements import parse_statement
from hailmary.allocation.returns import last_business_day_on_or_before
from hailmary.data.providers import YahooFinanceProvider

ETF_XLSX = Path('../../data/stashaway_etf_universe.xlsx')
STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
REPORT_PATH = Path('../../reports/etf_explorer.html')
START = date(2020, 1, 1)
END = last_business_day_on_or_before(date.today())
TARGET_ANN_RETURN = 0.05
print(f'window: {START}..{END}')

window: 2020-01-01..2026-05-25


## Load user's book (for correlation reference)

In [2]:
parsed = parse_statement(STATEMENT_PATH)
portfolios = [
    from_parsed(
        p,
        roles=ROLES[p.name],
        metadata={'management_fee_annual': MGMT_FEES_ANNUAL.get(p.name, 0.0)},
    )
    for p in parsed if p.name in ROLES
]
provider = YahooFinanceProvider()
fx_bars = provider.get_bars(['USDSGD=X'], START, END)
fx_series_usd_sgd = fx_bars.xs('USDSGD=X', level=0)['close']
print(f'{len(portfolios)} portfolios loaded')

2026-05-25 00:21:17.844 | DEBUG    | hailmary.allocation.statements:get:169 - Statement cache hit for 2026-04 StashAway Monthly Statement.pdf


2026-05-25 00:21:17.848 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=fa570043d3f4


15 portfolios loaded


## Build the explorer DataFrame

In [3]:
explorer_df = build_etf_explorer(
    ETF_XLSX,
    portfolios=portfolios,
    price_source=provider,
    fx_series_usd_sgd=fx_series_usd_sgd,
    start=START,
    end=END,
)
print(f'{len(explorer_df)} ETFs, {explorer_df["has_data"].sum()} with Yahoo data')
explorer_df.head(20)

2026-05-25 00:21:18.117 | INFO     | hailmary.allocation.etf_explorer:build_etf_explorer:177 - ETF Explorer: fetching 97 symbols from Yahoo…


2026-05-25 00:21:18.119 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=ead95bfc3ee5


2026-05-25 00:21:18.172 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=333aa9072c3e


2026-05-25 00:21:18.196 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=8ca47a4035ad


2026-05-25 00:21:18.205 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=b328b58cfa0a


...

2026-05-25 00:21:18.295 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=da20efc4ae97


2026-05-25 00:21:18.371 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=b328b58cfa0a


2026-05-25 00:21:18.393 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=e443f1c4bd4f


98 ETFs, 98 with Yahoo data


,asset_class,name,ticker,wrapper,fund_manager,has_data,n_days,cum_return_1M,cum_return_3M,ytd_return,...,ann_return_5Y,max_dd_5Y,vol_5Y,corr_n,corr_book_1M,corr_book_3M,corr_book_YTD,corr_book_1Y,corr_book_3Y,corr_book_5Y
0,All Country World,iShares MSCI ACWI UCITS ETF,ISAC.L,UCITS (LSE),iShares,True,1577,4.535634e-02,0.077197,0.101938,...,0.117310,-0.252319,0.155445,283,0.779695,0.639523,0.614925,0.392476,0.395192,0.395192
1,Artificial Intelligence,Xtrackers Artificial Intelligence & Big Data U...,XAID.L,?,Xtrackers,True,1577,1.350281e-01,0.249901,0.209038,...,0.138193,-0.390967,0.223843,283,0.552086,0.519595,0.476376,0.326529,0.333690,0.333690
2,Asia ex-Japan,iShares MSCI All Country Asia ex Japan ETF,AAXJ,US,iShares,True,1561,7.077155e-02,0.088757,0.242788,...,0.108974,-0.387568,0.197122,283,0.809347,0.707548,0.696305,0.679849,0.670588,0.670588
3,Asia High Yield USD Corporate Bonds *,iShares USD Asia High Yield Bond ETF,QL3.SI,SG,iShares,True,1562,-4.440892e-16,0.013327,0.020372,...,-0.025536,-0.414487,0.092206,277,-0.369451,0.037617,0.048883,0.090235,0.102932,0.102932
4,Australia,iShares MSCI Australia ETF,EWA,US,iShares,True,1561,-1.674068e-02,-0.022750,0.104461,...,0.109493,-0.238233,0.195863,283,0.797671,0.687178,0.677526,0.720964,0.711085,0.711085
5,Aerospace & Defense,Invesco Aerospace & Defense ETF,PPA,US,Invesco,True,1561,9.308160e-03,-0.048414,0.104134,...,0.220938,-0.188185,0.181397,283,0.676009,0.658129,0.657147,0.696776,0.668632,0.668632
6,Battery Value-chain,L&G Battery Value-Chain UCITS ETF,BATT.L,?,LGIM,True,1577,2.646962e-02,0.213343,0.336668,...,0.143440,-0.371618,0.254309,283,0.527712,0.557549,0.548036,0.375653,0.359610,0.359610
7,Biotechnology,iShares Biotechnology ETF,IBB,US,iShares,True,1561,-1.597391e-02,-0.035332,-0.012310,...,0.048200,-0.364711,0.218168,283,0.671189,0.736481,0.644844,0.550263,0.526870,0.526870
8,Bitcoin (Accredited Investors only),Fidelity® Wise Origin® Bitcoin Fund,FBTC,?,Fidelity,True,575,-2.497406e-02,0.175486,-0.062571,...,0.315487,-0.465991,0.504064,283,0.694011,0.823869,0.813159,0.675085,0.672236,0.672236
9,Blockchain,Invesco CoinShares Global Blockchain UCITS ETF,BCHN.L,UCITS (LSE),Invesco,True,1575,1.004095e-01,0.232304,0.201512,...,0.052735,-0.572249,0.384653,283,0.793537,0.644689,0.629714,0.368323,0.374758,0.374758


## Render HTML report

In [4]:
out = render_etf_explorer_report(
    explorer_df,
    REPORT_PATH,
    target_ann_return=TARGET_ANN_RETURN,
)
print(f'Wrote {out.resolve()}')

Wrote C:\Users\Dalva\src\project-hail-mary\reports\etf_explorer.html
